I think the fundamental idea is:

input representation -> black box -> a modified representation -> logits -> softmax

this step:

representation -> logits

is essentially based on "current info" -> make a "judgement" (a.k.a getting logits).

We can assume this step is **fixed**: it's just a linear projection, no innovation there.

So the juciy question is: how to get **better** "current info" to support the judgement making?

---

### Idea 1: just use MLP for LLM (A.k.a, **no black box**)

Just pass through input representation!

input length fixed, if not used up, 0 padding.

then go through 1 or multiple giant MLP -> logits -> softmax etc.

According to UAT, with enough hidden state and weights, this in theory kind of work, even it's stupid.

But why it's stupid/inefficient?

#### Key issue 1: didn't **explicitly** share the same rule across positions.

[A, B, C, 0, 0] → MLP → logits

say A could follow by B, ex. I like.

then the MLP may need to learn same patterns separately at different positions

```text
Position 1: A → B
Position 2: A → B
Position 3: A → B
Position 4: A → B
Position 5: A → B
```

instead of learn a general rule between 2 tokens

---

### Idea 2: RNN

Let's call a modified representation, h_t, and input x

h_t = tanh(W_x * x_t + W_h * h_(t-1) + B)

Say currently we have:

> I love teaching

Those 3 words representation is h_(t - 1)

Now the new word

> AI

comes in

How do we get representation for

> I love teaching AI

a.k.a h_t?

> we grab some info from the next token:
>
> W_x * x_t
>
> and we grab some info from what we already have:
>
> W_h * h_(t-1)
>
> and mix!

W_x and W_h stays the same, so same function applys to every position, this may forth the model to learn position independant rule.

#### Key issue

hidden state is the same size! The longer the context, the more compression happens! 

The model cannot directly look back at the **original information**. It must rely on what **survived** in its hidden state.

---

### Idea 3: Attention

#### Why not?
> kept the representations of previous tokens and let the current token decide which ones to read?

Instead of asking:

>"How do I squeeze everything I've seen into one memory?"

We could ask:

>"Given what I'm trying to predict right now, which parts of the past should I look at?"



#### Leads to attention

What's involved in calculating the final represenation?

MLP:
>x_1, x_2 ... x_t are involved

RNN:
>only x_t and h_(t-1), the previous representation, are directly involved

Attention:
>x_1, x_2 ... x_t are involved

Combine the best ideas of both! See basic_attention, will skip the details here


### Idea 4: Compare RNN vs. Mamba, deltanet, etc

They are all trying to answer this question:

>If we're forced to keep a fixed-size memory, what's the most powerful update rule we can design?
>
>Core: h_t = f(h_t-1, x_t)

There are 3 sub-questions:

1. Storage: how to store memory, vector, matrix, or?
2. Write/update: when new info come, how to update memory?
3. Read: how to read info from memory?


### The Trade-off: Hoarder vs. Minimalist

**The Hoarder (ex. attention)**

> Keeps everything in a giant house. Nothing gets thrown away, but finding what you need can get expensive, and important information can get buried in all the clutter.

**The Minimalist (ex. Mamba)**

> Only allows one fixed-size box. Carefully decides what to keep and what to throw away. Super efficient — until you realize you threw away exactly what you need! 😂

Million dollar question for a house organizer:

> How do you know what to keep 😂？
>
> The Hoarder: postpones the decision of what's relevant until retrieval time.
>
> The Minimalist: make decisions as information arrives.